# all-reduce-grad-sync — faded example 3: Complete the skip-None grad sync loop

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `all-reduce-grad-sync`. The last cell reports your progress on the `Distributed: all_reduce grad sync` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: all_reduce grad sync` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`all-reduce-grad-sync`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "all-reduce-grad-sync"
DD_SUBTOPIC = "Distributed: all_reduce grad sync"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Frozen or unused parameters have `p.grad is None` and must be skipped, because calling all_reduce on a missing grad would error and, worse, desynchronize the collective across ranks. The standard guard is `if p.grad is None: continue` placed before the all_reduce + mean divide.

## Faded exercise 3

`sync_grads` must average grads over a model's parameters while skipping any parameter whose `.grad` is `None`. The all_reduce (a mock SUM) and the mean divide are written for the non-None case. Complete the body so None-grad parameters are skipped and the function returns the count of parameters actually synced.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import numpy as np
import torch as t
import torch.nn as nn

t.manual_seed(0)
world_size = 2

class MockDist:
    class ReduceOp:
        SUM = 'sum'
    @staticmethod
    def all_reduce(tensor, op):
        return tensor

def sync_grads(model, dist_module, world_size):
    synced = 0
    for p in model.parameters():
        if p.grad is None:
            continue
        dist_module.all_reduce(p.grad, op=dist_module.ReduceOp.SUM)
        p.grad /= world_size
        synced += 1
    return synced

model = nn.Linear(2, 1)
for p in model.parameters():
    p.grad = None
list(model.parameters())[0].grad = t.tensor([[4.0, 4.0]])  # only weight has a grad
n_synced = sync_grads(model, MockDist, world_size)


def _test():
    import torch as t
    assert n_synced == 1, n_synced
    w = list(model.parameters())[0]
    b = list(model.parameters())[1]
    assert t.allclose(w.grad, t.tensor([[2.0, 2.0]])), w.grad  # 4/world_size
    assert b.grad is None, 'bias had no grad and must stay None'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import numpy as np
import torch as t
import torch.nn as nn

t.manual_seed(0)
world_size = 2

class MockDist:
    class ReduceOp:
        SUM = 'sum'
    @staticmethod
    def all_reduce(tensor, op):
        return tensor

def sync_grads(model, dist_module, world_size):
    synced = 0
    for p in model.parameters():
        if p.grad is None:
            continue
        dist_module.all_reduce(p.grad, op=dist_module.ReduceOp.SUM)
        p.grad /= world_size
        synced += 1
    return synced

model = nn.Linear(2, 1)
for p in model.parameters():
    p.grad = None
list(model.parameters())[0].grad = t.tensor([[4.0, 4.0]])  # only weight has a grad
n_synced = sync_grads(model, MockDist, world_size)
```
</details>